# Inference-only follow-ups: scaled DPAR

This notebook **does not train or update any denoiser weights**. It reuses the preserved Gaussian and mixed checkpoints from the completed repair suite.

Goals:
1. calibrate correction scale $\beta$ on calibration prompts;
2. separate correction magnitude from DPAR geometry;
3. rescue-test mixed/structured DPAR with smaller correction;
4. rerun a dense $\alpha$ grid on held-out prompts to remove the coarse-grid ambiguity.

In [ ]:
import os, pathlib, subprocess, sys
repo = pathlib.Path('/content/steering-manifold-repair')
assert repo.exists(), 'Use the preserved Colab runtime where the repository already exists.'
subprocess.run(['git','-C',str(repo),'pull','--ff-only'], check=True)
os.chdir(repo)
subprocess.run([sys.executable,'-m','pip','install','-e','.'], check=True)
print('cwd:', os.getcwd())

## 0. Verify preserved artifacts

No training should be necessary. Stop if a checkpoint is missing.

In [ ]:
from pathlib import Path
required = [
    Path('results/sentiment_direction.pt'),
    Path('checkpoints/denoiser_gaussian.pt'),
    Path('checkpoints/denoiser_mixed.pt'),
]
for p in required:
    print(('OK ' if p.exists() else 'MISSING '), p)
assert all(p.exists() for p in required), 'A preserved checkpoint/direction is missing. Do not retrain yet; report which file is absent.'

In [ ]:
# Fast tests for the new inference-only code plus the original denoiser geometry tests.
subprocess.run([sys.executable,'-m','pytest','-q','tests/test_inference_followups.py','tests/test_denoiser.py'], check=True)

## 1. Archive figures from the completed experiments

This only copies existing runtime artifacts into the local `experiments/` tree; it does not rerun anything.

In [ ]:
subprocess.run([sys.executable,'scripts/archive_runtime_artifacts.py'], check=True)

## 2. Calibration sweep — no training

We independently choose $\beta$ for Gaussian vanilla, Gaussian DPAR, and mixed DPAR using only `data/calibration_prompts.txt`.
The denoiser weights remain frozen.

In [ ]:
subprocess.run([
    sys.executable, 'scripts/run_inference_followups.py',
    '--config', 'configs/inference_followups_gpt2.yaml',
    '--phase', 'calibration'
], check=True)

In [ ]:
import json, pandas as pd
from IPython.display import display, Image
selection = json.loads(Path('results/inference_followups/selection.json').read_text())
print(json.dumps(selection, indent=2))
display(pd.read_csv('results/inference_followups/calibration_beta_scores.csv'))

## 3. Held-out dense evaluation

Selected $\beta$ values are now frozen. Evaluation uses the original held-out prompts and seeds 11/23, with a denser $\alpha$ grid.

In [ ]:
subprocess.run([
    sys.executable, 'scripts/run_inference_followups.py',
    '--config', 'configs/inference_followups_gpt2.yaml',
    '--phase', 'evaluation'
], check=True)

In [ ]:
frontier = pd.read_csv('results/inference_followups/heldout_interpolated_frontier.csv')
display(frontier)
print(Path('results/inference_followups/SUMMARY.md').read_text())

In [ ]:
display(Image(filename='results/inference_followups/beta_calibration.png'))
display(Image(filename='results/inference_followups/selected_dense_pareto.png'))
display(Image(filename='results/inference_followups/selected_effective_alpha.png'))

## 4. Archive follow-up artifacts and package results

After this cell, send the ZIP back to ChatGPT for analysis. It contains only outputs/checkpoint metadata, not a retrained model.

In [ ]:
subprocess.run([sys.executable,'scripts/archive_runtime_artifacts.py','--include-followup'], check=True)
import shutil
archive = shutil.make_archive('/content/inference_followups_results', 'zip', 'results/inference_followups')
print('Created:', archive)